# CFD Simulation Assistant — Demo Notebook

This notebook walks through the main capabilities:
1. Parsing OpenFOAM case files
2. Analysing convergence logs
3. Searching the knowledge base
4. Running the full agent

No OpenFOAM installation required — we use the bundled sample cases.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

CASE_DIR = REPO_ROOT / "sample_cases" / "openfoam_dambreak"
LOG_PATH = CASE_DIR / "logs" / "interFoam.log"
FLUENT_LOG = REPO_ROOT / "sample_cases" / "fluent_limestone" / "convergence_log.txt"

print("Case dir exists:", CASE_DIR.exists())
print("Log exists:", LOG_PATH.exists())

## 1. Parse OpenFOAM controlDict

In [ ]:
from app.tools.parse_openfoam import parse_control_dict, explain_file

ctrl = parse_control_dict(CASE_DIR / "system" / "controlDict")
print("Parsed controlDict:")
for k, v in ctrl.items():
    print(f"  {k}: {v}")

In [ ]:
# Human-readable explanation
text = (CASE_DIR / "system" / "controlDict").read_text()
explanation = explain_file("controlDict", text)
print(explanation)

## 2. Analyse Convergence — OpenFOAM Log

In [ ]:
from app.tools.analyze_convergence import load_openfoam_log, full_analysis, plot_residuals

df = load_openfoam_log(LOG_PATH)
print(f"Parsed {len(df)} residual rows")
print(f"Fields: {df['field'].unique().tolist()}")
df.head(10)

In [ ]:
# Run full analysis
analysis = full_analysis(df)
print("Diverged:", analysis["diverged"])
print("Oscillating:", analysis["oscillating"])
print("Plateaued:", analysis["plateaued"])
print()
print("Findings:")
for exp in analysis["explanations"]:
    print(f"  {exp}")
print()
print("Suggested fixes:")
for i, fix in enumerate(analysis["suggestions"], 1):
    print(f"  {i}. {fix}")

In [ ]:
# Plot residuals
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("inline")
%matplotlib inline

plot_residuals(df, "/tmp/residuals.png", title="interFoam Dam-Break Residuals")
from IPython.display import Image
Image("/tmp/residuals.png")

## 3. Analyse Convergence — Fluent Limestone Contactor

In [ ]:
from app.tools.parse_fluent import parse_convergence_report

fluent_df = parse_convergence_report(FLUENT_LOG)
print(f"Fluent log: {len(fluent_df)} iterations, columns: {fluent_df.columns.tolist()}")
fluent_df.tail(10)

In [ ]:
# Visualise species plateau
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: continuity and velocity (converge nicely)
ax = axes[0]
for col in ["continuity", "x_velocity", "y_velocity", "z_velocity"]:
    if col in fluent_df.columns:
        ax.semilogy(fluent_df["iter"], fluent_df[col], label=col, lw=1.5)
ax.axhline(1e-6, color="green", ls="--", alpha=0.5, label="1e-6 target")
ax.set_xlabel("Iteration")
ax.set_ylabel("Residual")
ax.set_title("Continuity & Velocity Residuals")
ax.legend(fontsize=8)
ax.grid(which="both", alpha=0.3)

# Right: species (plateau at 1e-4)
ax = axes[1]
species_cols = [c for c in fluent_df.columns if c not in ("iter", "continuity", "x_velocity", "y_velocity", "z_velocity")]
for col in species_cols:
    ax.semilogy(fluent_df["iter"], fluent_df[col], label=col, lw=2)
ax.axhline(1e-4, color="red", ls="--", alpha=0.7, label="Plateau level (1e-4)")
ax.axhline(1e-6, color="green", ls="--", alpha=0.5, label="Target (1e-6)")
ax.set_xlabel("Iteration")
ax.set_ylabel("Residual")
ax.set_title("Species Residuals — Plateau Issue")
ax.legend(fontsize=8)
ax.grid(which="both", alpha=0.3)

plt.suptitle("Limestone Contactor — Convergence History", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("/tmp/fluent_convergence.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"Species plateau value: {fluent_df['co2'].iloc[-1]:.4e}")
print("Plateau NOT due to true convergence — need to diagnose source term coupling")

## 4. Generate Case Report (no LLM)

In [ ]:
from app.tools.generate_report import generate_openfoam_report
from IPython.display import Markdown

report = generate_openfoam_report(CASE_DIR)
Markdown(report)

## 5. Agent Demo (requires API key)

Set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` in your `.env` file first.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

if not os.getenv("OPENAI_API_KEY") and not os.getenv("ANTHROPIC_API_KEY"):
    print("No API key found. Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env to use the agent.")
else:
    from app.agent import CFDAssistant
    assistant = CFDAssistant(backend="openai")
    
    questions = [
        f"Analyze the convergence log at {LOG_PATH}",
        "What does maxCo = 0.9 mean for a VOF simulation and why is it too high?",
        "My species residuals in Fluent are stuck at 1e-4. What should I do?",
    ]
    
    for q in questions:
        print(f"\n{'='*60}")
        print(f"Q: {q}")
        print(f"{'='*60}")
        response = assistant.chat(q)
        print(response)